In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import find_dotenv
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

ROOT = Path(find_dotenv()).parent
df = pd.read_csv(ROOT / "data" / "processed" / "pm25_hourly_clean.csv")
df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Asia/Kolkata")

HORIZON = 24
LAGS = [1, 2, 3, 6, 12, 24, 48]

def mae(a, b):
    return np.mean(np.abs(a - b))

def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

results, importances = [], {}

for station, g in df.groupby("station"):
    s = g.sort_values("time").set_index("time")["pm25"]

    X = pd.DataFrame({"now": s})
    for k in LAGS:
        X[f"lag_{k}"] = s.shift(k)
    X["change_1h"] = s - s.shift(1)
    X["change_24h"] = s - s.shift(24)
    X["roll_mean_24"] = s.rolling(24).mean()
    X["roll_std_24"] = s.rolling(24).std()
    X["hour"] = X.index.hour
    X["target"] = s.shift(-HORIZON)

    data = X.dropna()
    if len(data) < 500:
        print("Skipping", station)
        continue

    split = int(len(data) * 0.8)
    train, test = data.iloc[:split], data.iloc[split:]
    features = [c for c in data.columns if c != "target"]
    print(station, "| train:", train.index.min().date(), "to", train.index.max().date(),
          "| test:", test.index.min().date(), "to", test.index.max().date())

    delta_train = train["target"] - train["now"]     # learn the CHANGE

    rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=20,
                               max_depth=8, n_jobs=-1, random_state=42)
    rf.fit(train[features], delta_train)

    xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3,
                       subsample=0.8, random_state=42)
    xgb.fit(train[features], delta_train)

    y = test["target"]
    p_base = test["now"]
    p_rf = np.clip(test["now"] + rf.predict(test[features]), 0, None)
    p_xgb = np.clip(test["now"] + xgb.predict(test[features]), 0, None)

    results.append({
        "station": station,
        "baseline_MAE": round(mae(y, p_base), 1),
        "RF_MAE": round(mae(y, p_rf), 1),
        "XGB_MAE": round(mae(y, p_xgb), 1),
        "baseline_RMSE": round(rmse(y, p_base), 1),
        "RF_RMSE": round(rmse(y, p_rf), 1),
        "XGB_RMSE": round(rmse(y, p_xgb), 1),
    })
    importances[station] = pd.Series(xgb.feature_importances_, index=features) \
                             .sort_values(ascending=False)

print()
print(pd.DataFrame(results).to_string(index=False))
first = next(iter(importances))
print("\nTop features for", first)
print(importances[first].head(6).round(3))

Anand Vihar, New Delhi - DPCC | train: 2025-10-17 to 2026-08-07 | test: 2026-08-07 to 2026-10-12
New Delhi | train: 2025-10-17 to 2026-06-18 | test: 2026-06-18 to 2026-08-17
Punjabi Bagh, Delhi - DPCC | train: 2025-10-21 to 2026-07-29 | test: 2026-07-29 to 2026-10-12
R K Puram, Delhi - DPCC | train: 2025-10-17 to 2026-07-22 | test: 2026-07-22 to 2026-10-11

                      station  baseline_MAE  RF_MAE  XGB_MAE  baseline_RMSE  RF_RMSE  XGB_RMSE
Anand Vihar, New Delhi - DPCC          30.2    25.0     24.2           40.6     33.4      32.8
                    New Delhi          14.6    14.5     14.2           19.5     18.2      18.1
   Punjabi Bagh, Delhi - DPCC          23.5    21.1     21.2           31.4     27.3      27.4
      R K Puram, Delhi - DPCC          20.5    19.1     19.2           27.4     24.9      25.1

Top features for Anand Vihar, New Delhi - DPCC
change_24h      0.209
now             0.145
roll_mean_24    0.102
lag_48          0.095
roll_std_24     0.071
lag_12 